# 07 — DiffQRCoder officiel, génération pas à pas et comparaison

Ce notebook exécute le code public de **jwliao1209/DiffQRCoder** au commit `e24ea73`, et non l'approximation img2img du projet. Il produit le QR de contrôle, le Stage 1 ControlNet, le Stage 2 SRPG, une variante SRPG + SR-MPGD, puis valide chaque image sur 13 dégradations × les décodeurs disponibles.

La priorité de sélection est stricte : **payload exact dans 100 % des validations**, puis CLIP-aesthetic, puis CLIPScore. Une image non stricte est conservée pour diagnostic, jamais marquée livrable.

## Chaîne réellement testée

```text
QR v3/M/mask 4 (740 px) ──► Stage 1 : Cetus-Mix + QR Monster v2
                                      │
                                      ├──► image artistique de référence
                                      │
même QR ──► bruit apparié ──► Stage 2 : nouveau débruitage DDIM + SRPG à chaque pas
                                      ├──► sortie SRPG
                                      └──► sortie SRPG + projection SR-MPGD finale

chaque sortie ──► 26 validations attendues ──► MER + CLIP-aesthetic + CLIPScore
             ──► porte 26/26 ──► livraison ou rejet explicite
```

Les deux variantes Stage 2 repartent du **même état aléatoire**, afin que la comparaison mesure SR-MPGD et non un changement de seed.

In [ ]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qrcode
import torch
from diffusers import ControlNetModel, DDIMScheduler
from IPython.display import Markdown, clear_output, display
from PIL import Image
from qrcode.exceptions import DataOverflowError

UPSTREAM_ROOT = Path('/opt/DiffQRCoder')
EXPECTED_COMMIT = 'e24ea73ee2e13c7e6e87cb422e8b11784e70ae00'
if not (UPSTREAM_ROOT / 'diffqrcoder' / 'pipeline_diffqrcoder.py').exists():
    raise RuntimeError('DiffQRCoder officiel absent de l image notebook : reconstruire Dockerfile.notebook')
sys.path.insert(0, str(UPSTREAM_ROOT))

from diffqrcoder import DiffQRCoderPipeline
from diffqrcoder.losses.perceptual_loss import PerceptualLoss
from prooftag_qr.qr import QRBlueprint, module_error_rate
from prooftag_qr.quality import image_change_metrics, image_quality_metrics
from prooftag_qr.quality_scoring import CLIPQualityScorer
from prooftag_qr.validation import QRValidator, summarize_validation_records

print('torch       :', torch.__version__)
print('diffusers   :', importlib.metadata.version('diffusers'))
print('transformers:', importlib.metadata.version('transformers'))
print('CUDA        :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'absent')
print('upstream    :', EXPECTED_COMMIT)
assert torch.cuda.is_available(), 'Ce notebook doit être exécuté dans le pod GPU, pas avec Python Windows.'
assert importlib.metadata.version('diffusers') == '0.32.2'

# Correctif minimal et audité : le dépôt construit une nouvelle constante avec torch.tensor,
# ce qui détache les quatre pertes VGG (et peut les replacer sur CPU). torch.stack conserve
# le device et le graphe. La formule, les poids et l'échelle restent strictement inchangés.
def differentiable_perceptual_forward(self, x, y):
    losses = [torch.nn.functional.mse_loss(fx, fy) for fx, fy in zip(self.extractor(x), self.extractor(y))]
    return torch.stack(losses).mean()

PerceptualLoss.forward = differentiable_perceptual_forward
UPSTREAM_PATCHES = [{
    'file': 'diffqrcoder/losses/perceptual_loss.py',
    'reason': 'préserver le gradient et le device de la perte perceptuelle',
    'change': 'torch.mean(torch.tensor(losses)) -> torch.stack(losses).mean()',
}]
print('Correctif amont audité appliqué :', UPSTREAM_PATCHES[0]['change'])


## 1. Paramètres modifiables

Commencer avec le payload témoin publié. Pour un vrai lien Prooftag, utiliser une URL courte qui tient obligatoirement dans un QR version 3/M ; la cellule suivante refusera toute surcharge au lieu de changer silencieusement de version.

In [ ]:
EXPERIMENT_NAME = 'e010-diffqrcoder-official-v1'
PAYLOAD = 'Thanks reviewer!'  # remplacer ensuite par une URL Prooftag courte
PROMPT = 'Winter wonderland, fresh snowfall, evergreen trees, cozy log cabin, smoke rising from chimney, aurora borealis in night sky.'
NEGATIVE_PROMPT = 'easynegative'
SEED = 1

QR_VERSION = 3
QR_ERROR_CORRECTION = qrcode.constants.ERROR_CORRECT_M
QR_MASK_PATTERN = 4
QR_MODULE_SIZE = 20
QR_BORDER_MODULES = 4
QR_INTERNAL_PADDING = 78  # 740 px est ramené à 736 par la VAE; crop 78 => cœur 580 = 29×20

BASE_MODEL_URL = 'https://huggingface.co/fp16-guy/Cetus-Mix_Whalefall_fp16_cleaned/blob/main/cetusMix_Whalefall2_fp16.safetensors'
CONTROLNET_MODEL = 'monster-labs/control_v1p_sd15_qrcode_monster'
CONTROLNET_SUBFOLDER = 'v2'
STEPS = 40
GUIDANCE_SCALE = 7.5
CONTROLNET_SCALE = 1.35
ETA = 0.0
PREVIEW_EVERY = 5
MEMORY_PROFILE = 'rtx_20gb'

STAGE2_PROFILES = [
    {'name': 'srpg', 'srg': 500, 'pg': 3, 'srmpgd_iterations': None, 'srmpgd_lr': 0.1},
    {'name': 'srpg_plus_srmpgd', 'srg': 500, 'pg': 3, 'srmpgd_iterations': 20, 'srmpgd_lr': 0.1},
]

run_name = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{EXPERIMENT_NAME}-seed{SEED}"
RUN_DIR = Path('/data/notebook-runs') / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Résultats :', RUN_DIR)


## 2. QR de contrôle strictement conforme au protocole

Version 3, correction M, masque 4, 4 modules de quiet zone, modules de 20 px. Aucun `fit=True` : si le contenu est trop long, l'expérience s'arrête et demande une URL plus courte.

In [ ]:
qr = qrcode.QRCode(
    version=QR_VERSION,
    error_correction=QR_ERROR_CORRECTION,
    box_size=QR_MODULE_SIZE,
    border=QR_BORDER_MODULES,
    mask_pattern=QR_MASK_PATTERN,
)
qr.add_data(PAYLOAD)
try:
    qr.make(fit=False)
except DataOverflowError as exc:
    raise ValueError(
        'Payload trop long pour QR v3/M. Utiliser une URL Prooftag courte; ne pas laisser la version varier.'
    ) from exc

qr_image = qr.make_image(fill_color='black', back_color='white').convert('RGB')
matrix = np.asarray(qr.get_matrix(), dtype=np.uint8)
blueprint = QRBlueprint(image=qr_image, matrix=matrix, version=QR_VERSION, border=QR_BORDER_MODULES)
qr_path = RUN_DIR / '00_qr_control.png'
qr_image.save(qr_path)
payload_hash = hashlib.sha256(PAYLOAD.encode('utf-8')).hexdigest()
print(f'Taille={qr_image.size}, matrice={matrix.shape}, SHA-256 payload={payload_hash}')
assert qr_image.size == (740, 740) and matrix.shape == (37, 37)
display(qr_image.resize((370, 370)))


## 3. Chargement de la fondation et du ControlNet

La première exécution télécharge Cetus-Mix Whalefall et QR Code Monster v2 dans le PVC `/cache`. Le scheduler est DDIM comme dans le code public. La RTX 4000 Ada 20 Go reste propriétaire exclusive du pod notebook.

In [ ]:
def cuda_memory_gib():
    return {
        'allocated': torch.cuda.memory_allocated() / 2**30,
        'reserved': torch.cuda.memory_reserved() / 2**30,
        'free_driver': torch.cuda.mem_get_info()[0] / 2**30,
        'total_driver': torch.cuda.mem_get_info()[1] / 2**30,
    }


def release_previous_gpu_objects():
    # empty_cache ne suffit pas si le kernel conserve encore une pipeline ou ses sorties.
    names = [
        'quality_scorer', 'candidates', 'stage2_results', 'stage1_output',
        'stage1_tensor', 'stage1_image', 'controlnet', 'pipe',
    ]
    objects = []
    for name in names:
        value = globals().pop(name, None)
        if value is not None:
            objects.append(value)
    for value in reversed(objects):
        if hasattr(value, 'to'):
            try:
                value.to('cpu')
            except Exception:
                pass
    objects.clear()
    shell = get_ipython()
    if shell is not None:
        output_history = shell.user_ns.get('Out')
        if isinstance(output_history, dict):
            output_history.clear()
        for history_name in ('_', '__', '___'):
            shell.user_ns.pop(history_name, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


print('VRAM avant nettoyage :', cuda_memory_gib())
release_previous_gpu_objects()
memory_after_cleanup = cuda_memory_gib()
print('VRAM après nettoyage :', memory_after_cleanup)
if memory_after_cleanup['allocated'] > 1.0:
    raise RuntimeError(
        'Le kernel conserve plus de 1 Gio CUDA après nettoyage. Utiliser Kernel > Restart Kernel, puis Run All Cells.'
    )
if memory_after_cleanup['free_driver'] < 15.0:
    raise RuntimeError(
        'Moins de 15 Gio sont libres au niveau du pilote. Vérifier nvidia-smi et arrêter tout autre consommateur GPU.'
    )

load_started = time.perf_counter()
controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_MODEL,
    subfolder=CONTROLNET_SUBFOLDER,
    torch_dtype=torch.float16,
    cache_dir='/cache/huggingface',
)
pipe = DiffQRCoderPipeline.from_single_file(
    BASE_MODEL_URL,
    controlnet=controlnet,
    torch_dtype=torch.float16,
    cache_dir='/cache/huggingface',
    safety_checker=None,
    use_safetensors=True,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
pipe.unet.requires_grad_(False).eval()
pipe.controlnet.requires_grad_(False).eval()
pipe.vae.requires_grad_(False).eval()
pipe.text_encoder.requires_grad_(False).eval()
pipe.enable_attention_slicing('max')
pipe.enable_vae_slicing()
if MEMORY_PROFILE == 'rtx_20gb':
    pipe.unet.enable_gradient_checkpointing()
    pipe.controlnet.enable_gradient_checkpointing()
print(f'Pipeline chargée en {time.perf_counter() - load_started:.1f}s; scheduler={type(pipe.scheduler).__name__}')
print('Profil mémoire :', MEMORY_PROFILE)
print('VRAM après chargement :', cuda_memory_gib())


## 4. Stage 1 — génération artistique ControlNet

Cette image est la référence perceptuelle du Stage 2. Elle est aussi mesurée seule : le papier indique environ 60 % de réussite sans le raffinement, mais ce taux doit être recalculé sur notre protocole.

In [ ]:
generator = torch.Generator(device='cuda').manual_seed(SEED)
stage1_started = time.perf_counter()
stage1_output = pipe._run_stage1(
    prompt=PROMPT,
    qrcode=qr_image,
    negative_prompt=NEGATIVE_PROMPT,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE_SCALE,
    eta=ETA,
    generator=generator,
    controlnet_conditioning_scale=CONTROLNET_SCALE,
    output_type='pt',
)
stage1_seconds = time.perf_counter() - stage1_started
stage1_tensor = stage1_output.images.detach()
stage1_image = pipe.image_processor.numpy_to_pil(pipe.image_processor.pt_to_numpy(stage1_tensor))[0]
stage1_image.save(RUN_DIR / '01_stage1_controlnet.png')
stage2_rng_state = generator.get_state().clone()
print(f'Stage 1 terminé en {stage1_seconds:.1f}s; image={stage1_image.size}; MER={module_error_rate(stage1_image, blueprint):.4f}')
display(stage1_image.resize((512, 512)))


## 5. Stage 2 — raffinement guidé observable

À chaque pas DDIM, le code officiel décode une estimation de l'image, calcule la Scanning Robust Loss sur les centres de modules, ajoute son gradient au bruit prédit et conserve la proximité perceptuelle avec le Stage 1. Les aperçus sont enregistrés tous les 5 pas. SR-MPGD, lorsqu'il est activé, optimise ensuite le latent final.

In [ ]:
def decode_latents(pipeline, latents):
    with torch.no_grad():
        decoded = pipeline.vae.decode(
            latents.detach() / pipeline.vae.config.scaling_factor, return_dict=False
        )[0]
        return pipeline.image_processor.postprocess(decoded, output_type='pil')[0]


def make_preview_callback(profile_name):
    preview_dir = RUN_DIR / f'previews_{profile_name}'
    preview_dir.mkdir()
    trace = []
    started = time.perf_counter()

    def callback(pipeline, step_index, timestep, callback_kwargs):
        if step_index % PREVIEW_EVERY == 0 or step_index == STEPS - 1:
            preview = decode_latents(pipeline, callback_kwargs['latents'])
            mer = module_error_rate(preview, blueprint)
            row = {
                'step': int(step_index),
                'timestep': int(timestep),
                'elapsed_s': round(time.perf_counter() - started, 3),
                'module_error_rate': mer,
            }
            trace.append(row)
            preview.save(preview_dir / f'step_{step_index:03d}.png')
            clear_output(wait=True)
            display(Markdown(f"**{profile_name} — pas {step_index + 1}/{STEPS} — MER observée {mer:.3%}**"))
            display(preview.resize((480, 480)))
        return callback_kwargs

    return callback, trace


@torch.no_grad()
def run_stage2(profile):
    paired_generator = torch.Generator(device='cuda')
    paired_generator.set_state(stage2_rng_state.clone())
    callback, trace = make_preview_callback(profile['name'])
    started = time.perf_counter()
    result = pipe._run_stage2(
        prompt=PROMPT,
        qrcode=qr_image,
        qrcode_module_size=QR_MODULE_SIZE,
        qrcode_padding=QR_INTERNAL_PADDING,
        ref_image=stage1_tensor,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        eta=ETA,
        generator=paired_generator,
        controlnet_conditioning_scale=CONTROLNET_SCALE,
        scanning_robust_guidance_scale=profile['srg'],
        perceptual_guidance_scale=profile['pg'],
        srmpgd_num_iteration=profile['srmpgd_iterations'],
        srmpgd_lr=profile['srmpgd_lr'],
        callback_on_step_end=callback,
        callback_on_step_end_tensor_inputs=['latents'],
        output_type='pil',
    )
    image = result.images[0]
    duration = time.perf_counter() - started
    image.save(RUN_DIR / f"02_{profile['name']}.png")
    (RUN_DIR / f"trace_{profile['name']}.json").write_text(json.dumps(trace, indent=2), encoding='utf-8')
    return image, duration, trace


In [ ]:
stage2_results = {}
for profile in STAGE2_PROFILES:
    print('Démarrage :', profile)
    image, duration, trace = run_stage2(profile)
    stage2_results[profile['name']] = {'image': image, 'duration_s': duration, 'trace': trace, 'profile': profile}
    clear_output(wait=True)
    print(f"{profile['name']} terminé en {duration:.1f}s; MER finale={module_error_rate(image, blueprint):.4f}")
    display(image.resize((512, 512)))


## 6. Validation stricte et scores visuels

Chaque candidate passe le payload exact via OpenCV et ZBar, s'ils sont présents, sur 13 scénarios : original, JPEG, flou, luminosité, contraste, réduction, bruit, rotation, gain/perte de point et perspective. Avec deux décodeurs, la porte contient 26/26 tests.

In [ ]:
validator = QRValidator()
print('Décodeurs disponibles :', [decoder.name for decoder in validator.decoders])
candidates = {'stage1_controlnet': stage1_image, **{name: value['image'] for name, value in stage2_results.items()}}
durations = {'stage1_controlnet': stage1_seconds, **{name: value['duration_s'] for name, value in stage2_results.items()}}

quality_scorer = CLIPQualityScorer(Path('/cache'), device='cpu')
rows = []
all_validations = {}
quality_error = None
for name, image in candidates.items():
    records = validator.validate(image, PAYLOAD)
    all_validations[name] = [asdict(record) for record in records]
    passed = sum(record.exact_payload_match for record in records)
    summary = summarize_validation_records(records)
    try:
        quality = quality_scorer.score(image, PROMPT)
        quality_values = asdict(quality)
    except Exception as exc:
        quality_error = f'{type(exc).__name__}: {exc}'
        quality_values = {'clip_similarity': None, 'clip_score': None, 'clip_aesthetic': None}
    row = {
        'candidate': name,
        'passed': passed,
        'total': len(records),
        'pass_rate': passed / len(records),
        'strict_all': passed == len(records),
        'worst_decoder_pass_rate': summary['worst_decoder_pass_rate'],
        'worst_scenario_pass_rate': summary['worst_scenario_pass_rate'],
        'module_error_rate': module_error_rate(image, blueprint),
        'duration_s': durations[name],
        **quality_values,
        **image_quality_metrics(image),
        **image_change_metrics(image, stage1_image),
    }
    rows.append(row)
    print(
        f"{name:20s} scan={passed:2d}/{len(records):2d} strict={row['strict_all']} "
        f"MER={row['module_error_rate']:.3%} aes={row['clip_aesthetic']} clip={row['clip_score']}"
    )

(RUN_DIR / 'validations.json').write_text(json.dumps(all_validations, indent=2), encoding='utf-8')
if quality_error:
    (RUN_DIR / 'quality-scoring-error.txt').write_text(quality_error, encoding='utf-8')
    print('ATTENTION scores CLIP indisponibles :', quality_error)


## 7. Comparaison finale et décision de livraison

Le classement ne compense jamais une lecture manquée par un bon score esthétique. Parmi les seules images strictes, CLIP-aesthetic puis CLIPScore départagent les candidates. Si aucune image n'est stricte, le meilleur résultat est exporté sous le nom `NOT_DELIVERABLE`.

In [ ]:
def safe_score(value):
    return float(value) if value is not None else float('-inf')

ranked = sorted(
    rows,
    key=lambda row: (
        row['strict_all'], row['pass_rate'], row['worst_decoder_pass_rate'],
        safe_score(row['clip_aesthetic']), safe_score(row['clip_score'])
    ),
    reverse=True,
)
selected = ranked[0]
selected_image = candidates[selected['candidate']]
delivery_status = 'DELIVERABLE' if selected['strict_all'] else 'NOT_DELIVERABLE'
delivery_path = RUN_DIR / f"03_{delivery_status}_{selected['candidate']}.png"
selected_image.save(delivery_path)

with (RUN_DIR / 'comparison.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

fig = plt.figure(figsize=(16, 9))
grid = fig.add_gridspec(2, len(candidates), height_ratios=[4, 2])
for index, (name, image) in enumerate(candidates.items()):
    axis = fig.add_subplot(grid[0, index])
    axis.imshow(image)
    row = next(item for item in rows if item['candidate'] == name)
    axis.set_title(f"{name}\nscan {row['passed']}/{row['total']} | MER {row['module_error_rate']:.2%}")
    axis.axis('off')
axis = fig.add_subplot(grid[1, :])
names = [row['candidate'] for row in rows]
positions = np.arange(len(names))
axis.bar(positions - 0.18, [row['pass_rate'] for row in rows], width=0.36, label='taux de lecture')
axis.bar(positions + 0.18, [1 - row['module_error_rate'] for row in rows], width=0.36, label='1 - MER')
axis.axhline(1.0, color='red', linestyle='--', label='porte stricte')
axis.set_ylim(0, 1.05)
axis.set_xticks(positions, names)
axis.grid(axis='y', alpha=0.25)
axis.legend(loc='lower right')
fig.suptitle(f"DiffQRCoder officiel — décision {delivery_status}: {selected['candidate']}")
fig.tight_layout()
fig.savefig(RUN_DIR / 'comparison-final.png', dpi=160, bbox_inches='tight')
display(fig)

print('CLASSEMENT FINAL')
for position, row in enumerate(ranked, 1):
    print(f"{position}. {row['candidate']:20s} {row['passed']}/{row['total']} strict={row['strict_all']} aes={row['clip_aesthetic']} clip={row['clip_score']}")
print(f'\nDécision : {delivery_status} — {delivery_path.name}')


In [ ]:
manifest = {
    'experiment': EXPERIMENT_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'diffqrcoder_commit': EXPECTED_COMMIT,
    'payload_sha256': payload_hash,
    'prompt': PROMPT,
    'negative_prompt': NEGATIVE_PROMPT,
    'seed': SEED,
    'qr': {'version': QR_VERSION, 'error_correction': 'M', 'mask_pattern': QR_MASK_PATTERN, 'module_size': QR_MODULE_SIZE, 'border': QR_BORDER_MODULES, 'internal_padding': QR_INTERNAL_PADDING},
    'models': {'base': BASE_MODEL_URL, 'controlnet': CONTROLNET_MODEL, 'subfolder': CONTROLNET_SUBFOLDER},
    'diffusion': {'steps_per_stage': STEPS, 'guidance_scale': GUIDANCE_SCALE, 'controlnet_scale': CONTROLNET_SCALE, 'eta': ETA, 'scheduler': type(pipe.scheduler).__name__, 'memory_profile': MEMORY_PROFILE},
    'stage2_profiles': STAGE2_PROFILES,
    'software': {'torch': torch.__version__, 'diffusers': importlib.metadata.version('diffusers'), 'transformers': importlib.metadata.version('transformers')},
    'upstream_patches': UPSTREAM_PATCHES,
    'validation_count_per_candidate': rows[0]['total'],
    'results': rows,
    'selected_candidate': selected['candidate'],
    'delivery_status': delivery_status,
    'quality_scoring_error': quality_error,
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(RUN_DIR / 'upstream-patches.json').write_text(json.dumps(UPSTREAM_PATCHES, indent=2), encoding='utf-8')
shutil.copy2('/workspace/notebooks/07_diffqrcoder_official_live.ipynb', RUN_DIR / '07_diffqrcoder_official_live.ipynb')

with (RUN_DIR / 'physical-validation.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow(['candidate', 'device', 'medium', 'distance_cm', 'angle_deg', 'lighting', 'success', 'notes'])
    for device in ['Pixel 7', 'iPhone 13', 'autre téléphone']:
        for medium in ['écran', 'impression']:
            writer.writerow([selected['candidate'], device, medium, '', '', '', '', ''])

archive_path = Path(shutil.make_archive(str(RUN_DIR), 'gztar', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
archive_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Archive :', archive_path)
print('SHA-256:', archive_hash)
print('Pour la récupérer, exécuter d abord sur le serveur Linux :')
print("POD=$(kubectl get pod -n qr-core -l app=prooftag-qr-notebook -o jsonpath='{.items[0].metadata.name}')")
print(f'kubectl cp -n qr-core "${{POD}}:{archive_path}" "$HOME/{archive_path.name}"')
print('Puis depuis PowerShell sur le PC :')
print(f'scp paul@pcIA:~/{archive_path.name} "$HOME/Downloads/"')


## Interprétation correcte

- Un 26/26 signifie que cette image a franchi la porte logicielle locale ; ce n'est pas encore une garantie universelle.
- Le papier teste des QR courts version 3. Une URL longue change la densité et invalide la comparaison.
- Les taux publiés (jusqu'à 99–100 % selon l'ablation) proviennent de leur jeu de 100 prompts et de leur protocole. Ils ne sont pas recopiés comme résultat Prooftag.
- La validation physique doit être remplie sur plusieurs téléphones, distances, angles, éclairages et impressions.
- Le code public et le pseudo-code du papier présentent des écarts (notamment l'initialisation du Stage 2 et le détail de SR-MPGD). Le manifest conserve donc le commit exact réellement exécuté.